In [0]:
%pip install databricks-labs-dqx
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
default_database_name = "main"
default_schema_name = "default"

dbutils.widgets.text("demo_database_name", default_database_name, "Catalog Name")
dbutils.widgets.text("demo_schema_name", default_schema_name, "Schema Name")

database = dbutils.widgets.get("demo_database_name")
schema = dbutils.widgets.get("demo_schema_name")

print(f"Selected Catalog for Demo Dataset: {database}")
print(f"Selected Schema for Demo Dataset: {schema}")

sensor_table = f"{database}.{schema}.sensor_data"
maintenance_table = f"{database}.{schema}.maintenance_data"

Selected Catalog for Demo Dataset: dpx_demo
Selected Schema for Demo Dataset: dqx


In [0]:
%sql
CREATE TABLE IF NOT EXISTS dqx_demo.dqx.sensor_data (
    sensor_id STRING,
    machine_id STRING,
    sensor_type STRING,
    reading_value DOUBLE,
    reading_timestamp TIMESTAMP,
    calibration_date DATE,
    battery_level INT,
    facility_zone STRING,
    is_active BOOLEAN,
    firmware_version STRING,
    ingest_date DATE
)
USING DELTA;

No such comm: LSP_COMM_ID
No such comm: LSP_COMM_ID
No such comm: LSP_COMM_ID
No such comm: LSP_COMM_ID


In [0]:
from pyspark.sql import Row, SparkSession
from datetime import datetime, date

spark = SparkSession.builder.getOrCreate()

# Defining 20 rows of data manually using Row()
data = [
    Row("SN-101", "MAC-01", "Temp", 22.5, datetime(2024,1,1,12,0), date(2023,1,1), 90, "Zone-A", True, "v1", date.today()),
    Row("SN-102", "MAC-01", "Press", 101.2, datetime(2024,1,1,12,5), date(2023,1,1), 85, "Zone-A", True, "v1", date.today()),
    Row("SN-103", "MAC-02", "Vib", 0.02, datetime(2024,1,1,12,10), date(2023,2,1), 70, "Zone-B", True, "v2", date.today()),
    Row("SN-104", "MAC-03", "Temp", 24.8, datetime(2024,1,1,12,15), date(2023,3,1), 40, "Zone-C", False, "v1", date.today()),
    Row("SN-105", "MAC-01", "Humid", 45.0, datetime(2024,1,1,12,20), date(2023,1,1), 88, "Zone-A", True, "v1", date.today()),
    Row("SN-106", "MAC-02", "Temp", 21.0, datetime(2024,1,1,12,25), date(2023,2,1), 95, "Zone-B", True, "v2", date.today()),
    Row("SN-107", "MAC-04", "Press", 99.5, datetime(2024,1,1,12,30), date(2023,4,1), 100, "Zone-D", True, "v3", date.today()),
    Row("SN-108", "MAC-03", "Vib", 0.08, datetime(2024,1,1,12,35), date(2023,3,1), 30, "Zone-C", True, "v1", date.today()),
    Row("SN-109", "MAC-05", "Temp", 25.5, datetime(2024,1,1,12,40), date(2023,5,1), 80, "Zone-A", True, "v3", date.today()),
    Row("SN-110", "MAC-01", "Press", 102.1, datetime(2024,1,1,12,45), date(2023,1,1), 15, "Zone-A", False, "v1", date.today()),
    Row("SN-111", "MAC-02", "Humid", 50.5, datetime(2024,1,1,12,50), date(2023,2,1), 60, "Zone-B", True, "v2", date.today()),
    Row("SN-112", "MAC-04", "Temp", 22.9, datetime(2024,1,1,12,55), date(2023,4,1), 92, "Zone-D", True, "v3", date.today()),
    Row("SN-113", "MAC-05", "Vib", 0.01, datetime(2024,1,1,13,0), date(2023,5,1), 85, "Zone-A", True, "v3", date.today()),
    Row("SN-114", "MAC-01", "Temp", 23.0, datetime(2024,1,1,13,5), date(2023,1,1), 75, "Zone-A", True, "v1", date.today()),
    Row("SN-115", "MAC-03", "Press", 100.8, datetime(2024,1,1,13,10), date(2023,3,1), 50, "Zone-C", True, "v1", date.today()),
    Row("SN-116", "MAC-02", "Temp", 20.5, datetime(2024,1,1,13,15), date(2023,2,1), 91, "Zone-B", True, "v2", date.today()),
    Row("SN-117", "MAC-04", "Vib", 0.15, datetime(2024,1,1,13,20), date(2023,4,1), 10, "Zone-D", False, "v3", date.today()),
    Row("SN-118", "MAC-05", "Humid", 48.2, datetime(2024,1,1,13,25), date(2023,5,1), 78, "Zone-A", True, "v3", date.today()),
    Row("SN-119", "MAC-01", "Temp", 22.1, datetime(2024,1,1,13,30), date(2023,1,1), 82, "Zone-A", True, "v1", date.today()),
    Row("SN-120", "MAC-02", "Press", 101.5, datetime(2024,1,1,13,35), date(2023,2,1), 94, "Zone-B", True, "v2", date.today())
]

# Convert list of Rows to DataFrame
# We use the existing table schema to ensure data types match perfectly
target_table = "dqx_demo.dqx.sensor_data"
df = spark.createDataFrame(data, schema=spark.table(target_table).schema)

# Append data into the SQL table
df.write.format("delta").mode("append").saveAsTable(target_table)

print(f"Successfully inserted {df.count()} rows into {target_table}")

Successfully inserted 20 rows into dqx_demo.dqx.sensor_data


In [0]:
sensor_bronze_data=spark.read.table('dqx_demo.dqx.sensor_data')
print("===sensor data sample===")
display(sensor_bronze_data.limit(10))

===sensor data sample===


sensor_id,machine_id,sensor_type,reading_value,reading_timestamp,calibration_date,battery_level,facility_zone,is_active,firmware_version,ingest_date
SN-101,MAC-01,Temp,22.5,2024-01-01T12:00:00.000Z,2023-01-01,90,Zone-A,true,v1,2026-03-10
SN-102,MAC-01,Press,101.2,2024-01-01T12:05:00.000Z,2023-01-01,85,Zone-A,true,v1,2026-03-10
SN-103,MAC-02,Vib,0.02,2024-01-01T12:10:00.000Z,2023-02-01,70,Zone-B,true,v2,2026-03-10
SN-104,MAC-03,Temp,24.8,2024-01-01T12:15:00.000Z,2023-03-01,40,Zone-C,false,v1,2026-03-10
SN-105,MAC-01,Humid,45.0,2024-01-01T12:20:00.000Z,2023-01-01,88,Zone-A,true,v1,2026-03-10
SN-106,MAC-02,Temp,21.0,2024-01-01T12:25:00.000Z,2023-02-01,95,Zone-B,true,v2,2026-03-10
SN-107,MAC-04,Press,99.5,2024-01-01T12:30:00.000Z,2023-04-01,100,Zone-D,true,v3,2026-03-10
SN-108,MAC-03,Vib,0.08,2024-01-01T12:35:00.000Z,2023-03-01,30,Zone-C,true,v1,2026-03-10
SN-109,MAC-05,Temp,25.5,2024-01-01T12:40:00.000Z,2023-05-01,80,Zone-A,true,v3,2026-03-10
SN-110,MAC-01,Press,102.1,2024-01-01T12:45:00.000Z,2023-01-01,15,Zone-A,false,v1,2026-03-10


In [0]:
%sql
CREATE TABLE IF NOT EXISTS dqx_demo.dqx.maintenance_data (
    maintenance_id STRING,
    machine_id STRING,
    maintenance_type STRING,
    maintenance_date DATE,
    duration_min INT,
    cost DOUBLE,
    next_schedule_date DATE,
    work_order_id STRING,
    safety_check_passed BOOLEAN,
    parts_list STRING,
    ingest_date DATE
)
USING DELTA;

In [0]:
from pyspark.sql import Row, SparkSession
from datetime import date, timedelta

spark = SparkSession.builder.getOrCreate()

# Defining 20 rows of maintenance data
data = [
    Row("M-1001", "MAC-01", "Routine", date(2024,1,5), 60, 150.0, date(2024,4,5), "WO-500", True, "Oil Filter", date.today()),
    Row("M-1002", "MAC-02", "Repair", date(2024,1,7), 120, 450.5, date(2024,7,7), "WO-501", True, "Bearing, Grease", date.today()),
    Row("M-1003", "MAC-03", "Inspection", date(2024,1,10), 45, 75.0, date(2024,2,10), "WO-502", True, "None", date.today()),
    Row("M-1004", "MAC-04", "Emergency", date(2024,1,12), 240, 1200.0, date(2024,4,12), "WO-503", False, "Motor, Belt", date.today()),
    Row("M-1005", "MAC-05", "Routine", date(2024,1,15), 60, 150.0, date(2024,4,15), "WO-504", True, "Sealant", date.today()),
    Row("M-1006", "MAC-01", "Calibration", date(2024,1,18), 90, 200.0, date(2024,7,18), "WO-505", True, "Sensor Kit", date.today()),
    Row("M-1007", "MAC-02", "Routine", date(2024,1,20), 65, 155.0, date(2024,4,20), "WO-506", True, "Oil Filter", date.today()),
    Row("M-1008", "MAC-03", "Repair", date(2024,1,22), 180, 600.0, date(2024,7,22), "WO-507", True, "Drive Chain", date.today()),
    Row("M-1009", "MAC-04", "Inspection", date(2024,1,25), 50, 80.0, date(2024,2,25), "WO-508", True, "None", date.today()),
    Row("M-1010", "MAC-05", "Routine", date(2024,1,28), 60, 150.0, date(2024,4,28), "WO-509", True, "Gasket", date.today()),
    Row("M-1011", "MAC-01", "Repair", date(2024,2,1), 150, 520.0, date(2024,8,1), "WO-510", True, "Wiring Harness", date.today()),
    Row("M-1012", "MAC-02", "Inspection", date(2024,2,3), 40, 75.0, date(2024,3,3), "WO-511", True, "None", date.today()),
    Row("M-1013", "MAC-03", "Routine", date(2024,2,5), 70, 160.0, date(2024,5,5), "WO-512", True, "Filter", date.today()),
    Row("M-1014", "MAC-04", "Calibration", date(2024,2,8), 100, 210.0, date(2024,8,8), "WO-513", True, "Nozzles", date.today()),
    Row("M-1015", "MAC-05", "Emergency", date(2024,2,10), 300, 1500.0, date(2024,5,10), "WO-514", False, "Main Board", date.today()),
    Row("M-1016", "MAC-01", "Routine", date(2024,2,12), 60, 150.0, date(2024,5,12), "WO-515", True, "Lubricant", date.today()),
    Row("M-1017", "MAC-02", "Routine", date(2024,2,15), 60, 150.0, date(2024,5,15), "WO-516", True, "Oil Filter", date.today()),
    Row("M-1018", "MAC-03", "Inspection", date(2024,2,18), 45, 80.0, date(2024,3,18), "WO-517", True, "None", date.today()),
    Row("M-1019", "MAC-04", "Repair", date(2024,2,20), 110, 380.0, date(2024,8,20), "WO-518", True, "Fuse, Switch", date.today()),
    Row("M-1020", "MAC-05", "Routine", date(2024,2,22), 65, 160.0, date(2024,5,22), "WO-519", True, "Seal", date.today())
]

# Convert and Append
target_table = "dqx_demo.dqx.maintenance_data"
df = spark.createDataFrame(data, schema=spark.table(target_table).schema)
df.write.format("delta").mode("append").saveAsTable(target_table)

print(f"Inserted {df.count()} rows into {target_table}")


Inserted 20 rows into dqx_demo.dqx.maintenance_data


In [0]:
import yaml
from pprint import pprint

from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.labs.dqx.profiler.generator import DQGenerator
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.config import WorkspaceFileChecksStorageConfig,TableChecksStorageConfig

In [0]:
mntnc_bronze_df=spark.read.table('dqx_demo.dqx.maintenance_data')
ws=WorkspaceClient()
dq_engine=DQEngine(ws)

In [0]:
profiler=DQProfiler(ws)
summary_stats,profiles=profiler.profile(mntnc_bronze_df)
display(summary_stats)

{'maintenance_id': {'count': 12,
  'mean': None,
  'stddev': None,
  'min': 'M-1001',
  '25%': None,
  '50%': None,
  '75%': None,
  'max': 'M-1020',
  'count_non_null': 12,
  'count_null': 0},
 'machine_id': {'count': 12,
  'mean': None,
  'stddev': None,
  'min': 'MAC-01',
  '25%': None,
  '50%': None,
  '75%': None,
  'max': 'MAC-05',
  'count_non_null': 12,
  'count_null': 0},
 'maintenance_type': {'count': 12,
  'mean': None,
  'stddev': None,
  'min': 'Calibration',
  '25%': None,
  '50%': None,
  '75%': None,
  'max': 'Routine',
  'count_non_null': 12,
  'count_null': 0},
 'duration_min': {'count': 12,
  'mean': 81.66666666666667,
  'stddev': 37.55803589920619,
  'min': 40,
  '25%': 60,
  '50%': 65,
  '75%': 90,
  'max': 180,
  'count_non_null': 12,
  'count_null': 0},
 'cost': {'count': 12,
  'mean': 218.375,
  'stddev': 150.37787630438922,
  'min': 75.0,
  '25%': 150.0,
  '50%': 160.0,
  '75%': 200.0,
  'max': 600.0,
  'count_non_null': 12,
  'count_null': 0},
 'work_order_id'

In [0]:
generator=DQGenerator(ws)
maintenance_checks=generator.generate_dq_rules(profiles)

13:33:59  INFO [d.l.d.profiler.generator] ✅ Quality rules generation completed. Generated 15 rules.


In [0]:
print("===Inferred DQ Rules ===\n")
for idx,check in enumerate(maintenance_checks):
    print(f"===========Check {idx} =========\n")
    pprint(check)

===Inferred DQ Rules ===

===========Check 0 =========

{'check': {'arguments': {'column': 'maintenance_id'},
           'function': 'is_not_null'},
 'criticality': 'error',
 'name': 'maintenance_id_is_null'}
===========Check 1 =========

{'check': {'arguments': {'column': 'machine_id'}, 'function': 'is_not_null'},
 'criticality': 'error',
 'name': 'machine_id_is_null'}
===========Check 2 =========

{'check': {'arguments': {'column': 'maintenance_type'},
           'function': 'is_not_null'},
 'criticality': 'error',
 'name': 'maintenance_type_is_null'}
===========Check 3 =========

{'check': {'arguments': {'column': 'maintenance_date'},
           'function': 'is_not_null'},
 'criticality': 'error',
 'name': 'maintenance_date_is_null'}
===========Check 4 =========

{'check': {'arguments': {'column': 'maintenance_date',
                         'max_limit': '2024-02-22',
                         'min_limit': '2024-01-05'},
           'function': 'is_in_range'},
 'criticality': 'error',

In [0]:
valid_df,quarantined_df=dq_engine.apply_checks_by_metadata_and_split(mntnc_bronze_df,maintenance_checks)
print("===Maintenance Bad Data Sample===")
display(quarantined_df)

===Maintenance Bad Data Sample===


maintenance_id,machine_id,maintenance_type,maintenance_date,duration_min,cost,next_schedule_date,work_order_id,safety_check_passed,parts_list,ingest_date,_errors,_warnings
M-1003,MAC-03,Inspection,2024-01-10,45,75.0,2024-02-10,WO-502,true,None,2026-03-10,"List(List(next_schedule_date_isnt_in_range, Value '2024-02-10' in Column 'next_schedule_date' not in range: [2024-03-03, 2024-08-08], List(next_schedule_date), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))",null
M-1004,MAC-04,Emergency,2024-01-12,240,1200.0,2024-04-12,WO-503,false,"Motor, Belt",2026-03-10,"List(List(duration_min_isnt_in_range, Value '240' in Column 'duration_min' not in range: [40, 180], List(duration_min), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()), List(cost_isnt_in_range, Value '1200.0' in Column 'cost' not in range: [75.0, 600.0], List(cost), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))",null
M-1009,MAC-04,Inspection,2024-01-25,50,80.0,2024-02-25,WO-508,true,None,2026-03-10,"List(List(next_schedule_date_isnt_in_range, Value '2024-02-25' in Column 'next_schedule_date' not in range: [2024-03-03, 2024-08-08], List(next_schedule_date), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))",null
M-1015,MAC-05,Emergency,2024-02-10,300,1500.0,2024-05-10,WO-514,false,Main Board,2026-03-10,"List(List(duration_min_isnt_in_range, Value '300' in Column 'duration_min' not in range: [40, 180], List(duration_min), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()), List(cost_isnt_in_range, Value '1500.0' in Column 'cost' not in range: [75.0, 600.0], List(cost), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))",null
M-1019,MAC-04,Repair,2024-02-20,110,380.0,2024-08-20,WO-518,true,"Fuse, Switch",2026-03-10,"List(List(next_schedule_date_isnt_in_range, Value '2024-08-20' in Column 'next_schedule_date' not in range: [2024-03-03, 2024-08-08], List(next_schedule_date), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))",null
M-1003,MAC-03,Inspection,2024-01-10,45,75.0,2024-02-10,WO-502,true,None,2026-03-10,"List(List(next_schedule_date_isnt_in_range, Value '2024-02-10' in Column 'next_schedule_date' not in range: [2024-03-03, 2024-08-08], List(next_schedule_date), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))",null
M-1004,MAC-04,Emergency,2024-01-12,240,1200.0,2024-04-12,WO-503,false,"Motor, Belt",2026-03-10,"List(List(duration_min_isnt_in_range, Value '240' in Column 'duration_min' not in range: [40, 180], List(duration_min), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()), List(cost_isnt_in_range, Value '1200.0' in Column 'cost' not in range: [75.0, 600.0], List(cost), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))",null
M-1009,MAC-04,Inspection,2024-01-25,50,80.0,2024-02-25,WO-508,true,None,2026-03-10,"List(List(next_schedule_date_isnt_in_range, Value '2024-02-25' in Column 'next_schedule_date' not in range: [2024-03-03, 2024-08-08], List(next_schedule_date), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))",null
M-1015,MAC-05,Emergency,2024-02-10,300,1500.0,2024-05-10,WO-514,false,Main Board,2026-03-10,"List(List(duration_min_isnt_in_range, Value '300' in Column 'duration_min' not in range: [40, 180], List(duration_min), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()), List(cost_isnt_in_range, Value '1500.0' in Column 'cost' not in range: [75.0, 600.0], List(cost), null, is_in_range, 2026-03-10T14:02:09.426Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))",null
M-1019,MAC-04,Repair,2024-02-20,110,380.0,2024-08-20,WO-518,true,"Fuse, Switch",2026-03-10,"List(List(next_schedu

In [0]:
sensor_dq_checks = yaml.safe_load("""
# Completeness Check on 2 columns
- criticality: error
  check:
    function: is_not_null_and_not_empty
    for_each_column:
    - sensor_id
    - machine_id
        
# Filter + Range Based Check
- criticality: warn
  filter: sensor_type = 'temperature'
  check:
    function: is_in_range
    arguments:
      column: reading_value
      min_limit: 0
      max_limit: 100

# Regex Based Check
- criticality: warn
  check:
    function: regex_match
    arguments:
      column: machine_id
      regex: '^MCH-\\d{3}$'

# timeliness check
- criticality: error
  check:
    function: is_not_in_future
    arguments:
      column: reading_timestamp
      offset: 259200

# sql_expression check
- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: (calibration_date > date(reading_timestamp))
      msg: Sensor calibration_date is later than sensor reading_timestamp
      name: calib_dt_gt_reading_ts
      negate: true
""")

# validate the checks
status = DQEngine.validate_checks(sensor_dq_checks)
print(status)

No errors found


In [0]:
sensor_bronze_df = spark.read.table('dqx_demo.dqx.sensor_data')

# Load quality rules from YAML file
# sensor_dq_checks = dq_engine.load_checks(config=WorkspaceFileChecksStorageConfig(location=sensor_dq_rules_yaml))

# Apply checks on input data
valid_df, quarantined_df = dq_engine.apply_checks_by_metadata_and_split(sensor_bronze_df, sensor_dq_checks)

print("=== Bad Data DF ===")
display(valid_df)
display(quarantined_df)

=== Bad Data DF ===


sensor_id,machine_id,sensor_type,reading_value,reading_timestamp,calibration_date,battery_level,facility_zone,is_active,firmware_version,ingest_date
SN-101,MAC-01,Temp,22.5,2024-01-01T12:00:00.000Z,2023-01-01,90,Zone-A,true,v1,2026-03-10
SN-102,MAC-01,Press,101.2,2024-01-01T12:05:00.000Z,2023-01-01,85,Zone-A,true,v1,2026-03-10
SN-103,MAC-02,Vib,0.02,2024-01-01T12:10:00.000Z,2023-02-01,70,Zone-B,true,v2,2026-03-10
SN-104,MAC-03,Temp,24.8,2024-01-01T12:15:00.000Z,2023-03-01,40,Zone-C,false,v1,2026-03-10
SN-105,MAC-01,Humid,45.0,2024-01-01T12:20:00.000Z,2023-01-01,88,Zone-A,true,v1,2026-03-10
SN-106,MAC-02,Temp,21.0,2024-01-01T12:25:00.000Z,2023-02-01,95,Zone-B,true,v2,2026-03-10
SN-107,MAC-04,Press,99.5,2024-01-01T12:30:00.000Z,2023-04-01,100,Zone-D,true,v3,2026-03-10
SN-108,MAC-03,Vib,0.08,2024-01-01T12:35:00.000Z,2023-03-01,30,Zone-C,true,v1,2026-03-10
SN-109,MAC-05,Temp,25.5,2024-01-01T12:40:00.000Z,2023-05-01,80,Zone-A,true,v3,2026-03-10
SN-110,MAC-01,Press,102.1,2024-01-01T12:45:00.000Z,2023-01-01,15,Zone-A,false,v1,2026-03-10


sensor_id,machine_id,sensor_type,reading_value,reading_timestamp,calibration_date,battery_level,facility_zone,is_active,firmware_version,ingest_date,_errors,_warnings
SN-101,MAC-01,Temp,22.5,2024-01-01T12:00:00.000Z,2023-01-01,90,Zone-A,true,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:18:12.701Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-102,MAC-01,Press,101.2,2024-01-01T12:05:00.000Z,2023-01-01,85,Zone-A,true,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:18:12.701Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-103,MAC-02,Vib,0.02,2024-01-01T12:10:00.000Z,2023-02-01,70,Zone-B,true,v2,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:18:12.701Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-104,MAC-03,Temp,24.8,2024-01-01T12:15:00.000Z,2023-03-01,40,Zone-C,false,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:18:12.701Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-105,MAC-01,Humid,45.0,2024-01-01T12:20:00.000Z,2023-01-01,88,Zone-A,true,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:18:12.701Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-106,MAC-02,Temp,21.0,2024-01-01T12:25:00.000Z,2023-02-01,95,Zone-B,true,v2,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:18:12.701Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-107,MAC-04,Press,99.5,2024-01-01T12:30:00.000Z,2023-04-01,100,Zone-D,true,v3,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:18:12.701Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-108,MAC-03,Vib,0.08,2024-01-01T12:35:00.000Z,2023-03-01,30,Zone-C,true,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:18:12.701Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-109,MAC-05,Temp,25.5,2024-01-01T12:40:00.000Z,2023-05-01,80,Zone-A,true,v3,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:18:12.701Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-110,MAC-01,Press,102.1,2024-01-01T12:45:00.000Z,2023-01-01,15,Zone-A,false,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:18:12.701Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"


In [0]:
df = dq_engine.apply_checks_by_metadata(sensor_bronze_df, sensor_dq_checks)
display(df)

sensor_id,machine_id,sensor_type,reading_value,reading_timestamp,calibration_date,battery_level,facility_zone,is_active,firmware_version,ingest_date,_errors,_warnings
SN-101,MAC-01,Temp,22.5,2024-01-01T12:00:00.000Z,2023-01-01,90,Zone-A,true,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:19:36.949Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-102,MAC-01,Press,101.2,2024-01-01T12:05:00.000Z,2023-01-01,85,Zone-A,true,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:19:36.949Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-103,MAC-02,Vib,0.02,2024-01-01T12:10:00.000Z,2023-02-01,70,Zone-B,true,v2,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:19:36.949Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-104,MAC-03,Temp,24.8,2024-01-01T12:15:00.000Z,2023-03-01,40,Zone-C,false,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:19:36.949Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-105,MAC-01,Humid,45.0,2024-01-01T12:20:00.000Z,2023-01-01,88,Zone-A,true,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:19:36.949Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-106,MAC-02,Temp,21.0,2024-01-01T12:25:00.000Z,2023-02-01,95,Zone-B,true,v2,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:19:36.949Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-107,MAC-04,Press,99.5,2024-01-01T12:30:00.000Z,2023-04-01,100,Zone-D,true,v3,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:19:36.949Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-108,MAC-03,Vib,0.08,2024-01-01T12:35:00.000Z,2023-03-01,30,Zone-C,true,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:19:36.949Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-109,MAC-05,Temp,25.5,2024-01-01T12:40:00.000Z,2023-05-01,80,Zone-A,true,v3,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:19:36.949Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
SN-110,MAC-01,Press,102.1,2024-01-01T12:45:00.000Z,2023-01-01,15,Zone-A,false,v1,2026-03-10,null,"List(List(machine_id_not_matching_regex, Column 'machine_id' is not matching regex, List(machine_id), null, regex_match, 2026-03-10T14:19:36.949Z, 01ce780b-8c25-44c9-9a13-63da2ae41631, Map()))"
